# Customer Shopping Behavior Analysis
**Independent portfolio project | Python (Pandas, Matplotlib)**

## Project objective
Explore customer demographics, product-category mix, purchase amounts, subscription status, discounts, review ratings and reported purchase frequency.

> **Dataset caveat:** This is a publicly available educational dataset, not proprietary company data. It contains 3,900 records and no transaction date. Treat the results as descriptive patterns in this dataset—not proof of real-world market behaviour, lifetime customer value, or month-by-month sales trends.

## Questions explored
1. What does the dataset contain, and where are the data-quality issues?
2. Which product categories have the largest recorded purchase-amount totals?
3. How do recorded purchase amounts compare between subscription groups?
4. What are the age and gender distributions?
5. How are discounts, seasons, shipping methods and purchase-frequency labels distributed?
6. What limitations should be considered before making business recommendations?


## 1. Import libraries and load the data

Place the original CSV in the `data/raw/` folder in the repository. If you run this notebook in Google Colab, upload the CSV when prompted.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("../data/raw/Costumer Shopping Behavior Dataset.csv")

if not DATA_PATH.exists():
    # Google Colab upload fallback
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError(
            f"Could not find {DATA_PATH}. Put the CSV in data/raw/ or update DATA_PATH."
        )

df_raw = pd.read_csv(DATA_PATH)
print("Dataset shape:", df_raw.shape)
display(df_raw.head())


## 2. Initial inspection

Check the schema, data types, missing values, duplicate rows and uniqueness of the customer identifier.

In [ ]:
df_raw.info()
display(df_raw.describe(include="all").T)
print("Duplicate rows:", df_raw.duplicated().sum())
print("Unique customer IDs:", df_raw["Customer ID"].nunique())
display(df_raw.isna().sum().rename("missing_values").to_frame())


## 3. Clean column names and standardize text

Column names are converted to lowercase `snake_case`. Text values are trimmed. The 37 missing review ratings are deliberately retained as missing; no ratings are invented.

In [ ]:
df = df_raw.copy()
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

print("Cleaned shape:", df.shape)
print("Missing review ratings:", df["review_rating"].isna().sum())
print("Duplicate rows after cleaning:", df.duplicated().sum())
display(df.head())


## 4. Data-quality checks

Check whether the discount and promo-code fields contain the same information. If they are identical, do not count them as two independent signals. Keep both in the cleaned source for transparency; document the redundancy for later analysis.

In [ ]:
discount_promo_match = (df["discount_applied"] == df["promo_code_used"]).mean()
print(f"Discount/promo agreement: {discount_promo_match:.1%}")
print("\nDiscount values:")
display(df["discount_applied"].value_counts(dropna=False).to_frame("records"))
print("\nPromo-code values:")
display(df["promo_code_used"].value_counts(dropna=False).to_frame("records"))


## 5. Descriptive summary

The sum of `purchase_amount_usd` is the sum of the recorded amounts in this file. Because the dataset has no transaction date or verified order history, it should not be described as company revenue or customer lifetime spend.

In [ ]:
total_recorded_amount = df["purchase_amount_usd"].sum()
summary = pd.DataFrame({
    "Metric": [
        "Records", "Unique customer IDs", "Total recorded purchase amount (USD)",
        "Average recorded purchase amount (USD)", "Median recorded purchase amount (USD)",
        "Average age", "Average review rating (available values)"
    ],
    "Value": [
        len(df), df["customer_id"].nunique(), total_recorded_amount,
        df["purchase_amount_usd"].mean(), df["purchase_amount_usd"].median(),
        df["age"].mean(), df["review_rating"].mean()
    ]
})
display(summary.round(2))


## 6. Category analysis

Compare record counts, summed recorded purchase amounts and average recorded purchase amounts by product category.

In [ ]:
category_summary = (
    df.groupby("category")["purchase_amount_usd"]
      .agg(records="count", recorded_amount_sum="sum", average_amount="mean")
      .sort_values("recorded_amount_sum", ascending=False)
)
display(category_summary.round(2))

ax = category_summary["recorded_amount_sum"].sort_values().plot(
    kind="barh", figsize=(8, 4), title="Sum of recorded purchase amounts by category"
)
ax.set_xlabel("USD (sum of records)")
ax.set_ylabel("Category")
plt.tight_layout()
plt.show()


## 7. Subscription comparison

Compare the recorded purchase amount by subscription status. This is a descriptive comparison only; it does not establish that subscriptions cause higher or lower spending.

In [ ]:
subscription_summary = (
    df.groupby("subscription_status")["purchase_amount_usd"]
      .agg(records="count", average_amount="mean", median_amount="median",
           recorded_amount_sum="sum")
)
display(subscription_summary.round(2))

ax = df.boxplot(column="purchase_amount_usd", by="subscription_status", figsize=(7, 4))
ax.set_title("Recorded purchase amount by subscription status")
ax.set_xlabel("Subscription status")
ax.set_ylabel("Purchase amount (USD)")
plt.suptitle("")
plt.tight_layout()
plt.show()


## 8. Customer age and gender distributions

Create broad age bands for descriptive reporting. These bands are analyst-defined, not official customer segments.

In [ ]:
age_bins = [17, 25, 35, 45, 55, 65, 100]
age_labels = ["18–25", "26–35", "36–45", "46–55", "56–65", "66+"]
df["age_group"] = pd.cut(df["age"], bins=age_bins, labels=age_labels)

age_summary = (
    df.groupby("age_group", observed=False)
      .agg(records=("customer_id", "count"),
           average_amount=("purchase_amount_usd", "mean"),
           recorded_amount_sum=("purchase_amount_usd", "sum"))
)
display(age_summary.round(2))
display(df["gender"].value_counts().rename("records").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["age_group"].value_counts(sort=False).plot(kind="bar", ax=axes[0], title="Records by age group")
df["gender"].value_counts().plot(kind="bar", ax=axes[1], title="Records by gender")
axes[0].set_xlabel("Age group")
axes[0].set_ylabel("Records")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Records")
plt.tight_layout()
plt.show()


## 9. Discount, season and purchase-frequency patterns

These charts show counts in the dataset. They do not show the effect of discounts or the timing of purchases over calendar time.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["discount_applied"].value_counts().plot(kind="bar", ax=axes[0], title="Discount applied")
df["season"].value_counts().plot(kind="bar", ax=axes[1], title="Season labels")
df["frequency_of_purchases"].value_counts().plot(kind="bar", ax=axes[2], title="Reported purchase frequency")
for ax in axes:
    ax.set_ylabel("Records")
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

discount_summary = df.groupby("discount_applied")["purchase_amount_usd"].agg(
    records="count", average_amount="mean", median_amount="median"
)
display(discount_summary.round(2))


## 10. Review-rating analysis

Missing ratings remain missing. Report the number of available ratings and summarize them without filling missing values.

In [ ]:
rating_summary = df["review_rating"].describe()
print("Missing ratings:", df["review_rating"].isna().sum())
display(rating_summary.to_frame("review_rating"))
ax = df["review_rating"].dropna().plot(
    kind="hist", bins=12, figsize=(7, 4), title="Distribution of available review ratings"
)
ax.set_xlabel("Review rating")
ax.set_ylabel("Records")
plt.tight_layout()
plt.show()


## 11. Key findings (generated from this dataset)

Run this cell to print the key descriptive results. Review each result and write your own interpretation before publishing the project.

In [ ]:
top_category = category_summary.index[0]
top_category_amount = category_summary.iloc[0]["recorded_amount_sum"]
subscription_means = df.groupby("subscription_status")["purchase_amount_usd"].mean()
top_gender = df["gender"].value_counts().idxmax()
top_season = df["season"].value_counts().idxmax()

print(f"Records: {len(df):,}")
print(f"Unique customer IDs: {df['customer_id'].nunique():,}")
print(f"Total recorded purchase amount: ${total_recorded_amount:,.0f}")
print(f"Average recorded purchase amount: ${df['purchase_amount_usd'].mean():.2f}")
print(f"Top category by summed recorded amount: {top_category} (${top_category_amount:,.0f})")
print(f"Most common gender in records: {top_gender}")
print(f"Most common season label: {top_season}")
print(f"Average amount - subscribers: ${subscription_means.get('Yes', float('nan')):.2f}")
print(f"Average amount - non-subscribers: ${subscription_means.get('No', float('nan')):.2f}")
print(f"Missing review ratings: {df['review_rating'].isna().sum()}")


## 12. Limitations and responsible interpretation

- The file has 3,900 rows and a customer ID field with 3,900 unique values; do not assume it is a multi-order transaction history.
- There is no transaction date. Month-over-month or year-over-year trends cannot be calculated.
- `purchase_amount_usd` is a recorded amount per row. Its sum is not verified business revenue or customer lifetime value.
- The dataset is publicly available and educational; it is not evidence from a named retailer's internal systems.
- The discount and promo-code columns may be redundant; inspect the equality check before using both as separate predictors.
- Correlation or group differences do not prove causation.
- Missing review ratings are left missing in this notebook.

## 13. Export the cleaned data

Save the cleaned file to the repository's `data/cleaned/` directory.

In [ ]:
output_dir = Path("../data/cleaned")
output_dir.mkdir(parents=True, exist_ok=True)
cleaned_path = output_dir / "customer_shopping_behavior_cleaned.csv"
df.to_csv(cleaned_path, index=False)
print(f"Saved cleaned dataset to: {cleaned_path}")
